# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [1]:
# Write your code below.

%load_ext dotenv
%dotenv ../01_materials/labs/.env



In [2]:
import dask.dataframe as dd

c:\Users\olgam\miniconda3\envs\dsi_participant\lib\site-packages\dask\dataframe\_pyarrow_compat.py:17: FutureWarning: Minimal version of pyarrow will soon be increased to 14.0.1. You are using 11.0.0. Please consider upgrading.
  warnings.warn(
C:\Users\olgam\AppData\Local\Temp\ipykernel_23188\676544213.py:1: DeprecationWarning: The current Dask DataFrame implementation is deprecated. 
In a future release, Dask DataFrame will use new implementation that
contains several improvements including a logical query planning.
The user-facing DataFrame API will remain unchanged.

The new implementation is already available and can be enabled by
installing the dask-expr library:

    $ pip install dask-expr

and turning the query planning option on:

    >>> import dask
    >>> dask.config.set({'dataframe.query-planning': True})
    >>> import dask.dataframe as dd

API documentation for the new implementation is available at
https://docs.dask.org/en/stable/dask-expr-api.html

Any feedback can be

+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [3]:
import os
from glob import glob
# Write your code below.

price_data = os.getenv("PRICE_DATA")
parquet_files = glob(os.path.join(price_data, "*/*.parquet"))
dd_p = dd.read_parquet(parquet_files).set_index("ticker")



In [4]:
dd_p

,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits,year
npartitions=49,,,,,,,,,
A,"datetime64[ns, America/New_York]",float64,float64,float64,float64,int64,float64,float64,int32
ABNB,...,...,...,...,...,...,...,...,...
...,...,...,...,...,...,...,...,...,...
YUM,...,...,...,...,...,...,...,...,...
YUM,...,...,...,...,...,...,...,...,...


For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Adjusted Close:
    
    - `returns`: (Adj Close / Adj Close_lag) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [5]:
# Write your code below.
import numpy as np
dd_feat = (
    dd_p.groupby('ticker', group_keys=False).apply(
        lambda x: x.assign(
            Close_lag_1 = x['Close'].shift(1),
            # conditions added to Adj_Close claculations to avoid division by zero 
            # by taking the Close value instead of the calculation
            Adj_Close = np.where(
                (x['Dividends'] == 0) & (x['Stock Splits'] == 0), 
                x['Close'], 
                x['Close'] * (1 - x['Dividends'].fillna(0)) / x['Stock Splits'].replace(0, 1)
            ),
            Adj_Close_lag_1 = np.where(
                (x['Dividends'] == 0) & (x['Stock Splits'] == 0), 
                x['Close'].shift(1), 
                (x['Close'] * (1 - x['Dividends'].fillna(0)) / x['Stock Splits'].replace(0, 1)).shift(1)
            )
        )
    ).assign(
        returns = lambda x: (x['Adj_Close'] / x['Adj_Close_lag_1']) - 1,
        hi_lo_range = lambda x: x['High'] - x['Low'])
            )


C:\Users\olgam\AppData\Local\Temp\ipykernel_23188\2709756852.py:4: UserWarning: `meta` is not specified, inferred from partial data. Please provide `meta` if the result is unexpected.
  Before: .apply(func)
  After:  .apply(func, meta={'x': 'f8', 'y': 'f8'}) for dataframe result
  or:     .apply(func, meta=('x', 'f8'))            for series result
  dd_p.groupby('ticker', group_keys=False).apply(


+ Convert the Dask data frame to a pandas data frame. 
+ Add a rolling average return calculation with a window of 10 days.
+ *Tip*: Consider using `.rolling(10).mean()`.

(3 pt)

In [6]:
# Write your code below.

dd_feat_pd = dd_feat.compute()
dd_feat_pd = dd_feat_pd.reset_index()
dd_feat_pd['rolling_avg_return'] = dd_feat_pd.groupby('ticker')['returns'].rolling(10).mean().reset_index(level=0, drop=True)
print(dd_feat_pd[['ticker', 'returns', 'rolling_avg_return']].head(11))

# That was probably unnesesssary to convert to pandas because we have previously found the size of the 
# csv and parquet files are quite small, so the dataframes should be relatively small as well, 
# and our simple calculation could be made directly in Dask without problems for RAM. 

# In this particular case with the small data frame (or unknown size) and the easy calculation, 
# that could be better to stay in dask. However, if we need to perform calculations requiring advanced aggregations,
# reshaping, time conversions for time-series data, easier handling of missing data, and easier data transformation,  
# this small dataset is better to convert to pandas, because the aforementioned funcitonalities in dask 
# are either limited or harder to use.

   ticker   returns  rolling_avg_return
0       A       NaN                 NaN
1       A -0.005262                 NaN
2       A  0.012280                 NaN
3       A -0.001493                 NaN
4       A  0.024860                 NaN
5       A  0.004194                 NaN
6       A  0.005086                 NaN
7       A -0.009036                 NaN
8       A  0.007659                 NaN
9       A -0.001810                 NaN
10      A  0.001451            0.003793


Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?
+ Would it have been better to do it in Dask? Why?

(1 pt)

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.